# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ishkaklotar03/ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**My rule in plain words (one rule, one reason code):**
A page is a **stale visible page** when it has not been updated for at least 180 days AND it still gets at least 500 impressions in 90 days. These pages get the reason code `stale_visible_page` and the action `REFRESH_REVIEW`. All other pages get `no_flag` and `MONITOR`. The score is 50% how visible the page is (percentile of `impressions_90d`) and 50% how stale it is (percentile of `days_since_last_update`).

**Why I lean on these two signals (both linked to FlyRank flags):**
- **Staleness** is the signal behind the refresh flag. Assumption: pages that have not been updated for longer are more often losing visibility.
- **Volume** is the signal behind the quick-win logic. Assumption: pages with more impressions have more at stake, and their trend is less noisy.

**What I check first:** one bucket table per signal, with n printed. I use `trend_direction == "down"` only as a proxy outcome to test the assumption. It is a bucket from the current window, not a future outcome, so every verdict is directional. It is never an input to the score.

**How the verdict word is chosen (by the code):** CONFIRMED = the share of "down" pages rises as the signal rises. OPPOSITE = it falls. MIXED = it goes up and down. FALSE = it is flat (spread under 3 points). Buckets with fewer than 30 pages are ignored for the verdict.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Clone the repo in Colab if the data is not already there
if (not Path("ML-Internship").exists()
        and not Path("data/raw/content_refresh_anonymized.csv").exists()
        and not Path("../../data/raw/content_refresh_anonymized.csv").exists()):
    !git clone https://github.com/ishkaklotar03/ML-Internship.git

candidates = [
    (Path("ML-Internship"), Path("ML-Internship/data/raw/content_refresh_anonymized.csv")),
    (Path("../.."), Path("../../data/raw/content_refresh_anonymized.csv")),
    (Path("."), Path("data/raw/content_refresh_anonymized.csv")),
]
ROOT, DATA_PATH = next(((r, p) for r, p in candidates if p.exists()), (None, None))
if DATA_PATH is None:
    raise FileNotFoundError("content_refresh_anonymized.csv not found")

df = pd.read_csv(DATA_PATH)

needed = ["content_id", "client_id", "impressions_90d", "sessions_90d",
          "content_age_days", "days_since_last_update", "avg_position", "trend_direction"]
missing = [c for c in needed if c not in df.columns]
if missing:
    print("Missing columns:", missing)
    print("Available columns:", list(df.columns))
    raise KeyError(f"Missing columns: {missing}")

# Same eligibility rule as the starter pipeline
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
df = df.drop_duplicates(subset="content_id").reset_index(drop=True)

# Proxy outcome, used ONLY to test the two signals (never in the score)
df["outcome_down"] = (df["trend_direction"] == "down").astype(int)
print(f"Pages: {len(df):,} | Clients: {df['client_id'].nunique()} | Overall share down: {df['outcome_down'].mean():.3f}\n")


def verdict_from(table, min_n=30, flat_gap=0.03, tol=0.01):
    t = table[table["n"] >= min_n]
    rates = t["share_down"].tolist()
    if len(rates) < 2:
        return "MIXED"  # not enough big buckets to judge
    if max(rates) - min(rates) < flat_gap:
        return "FALSE"
    diffs = np.diff(rates)
    if (diffs >= -tol).all():
        return "CONFIRMED"
    if (diffs <= tol).all():
        return "OPPOSITE"
    return "MIXED"


def run_signal(name, series, bins, labels):
    bucket = pd.cut(series, bins=bins, labels=labels, right=False)
    t = (pd.DataFrame({"bucket": bucket, "outcome_down": df["outcome_down"]})
         .dropna(subset=["bucket"])
         .groupby("bucket", observed=True)["outcome_down"]
         .agg(n="size", share_down="mean")
         .reset_index())
    t["share_down"] = t["share_down"].round(3)
    v = verdict_from(t)
    print(f"--- {name} ---")
    print(t.to_string(index=False))
    print(f"n used: {int(t['n'].sum()):,} of {len(df):,} pages")
    print(f"VERDICT: {v}\n")
    return {"verdict": v, "n_used": int(t["n"].sum())}


results = {}
results["staleness"] = run_signal(
    "Signal 1: staleness (days_since_last_update) vs share trending down",
    df["days_since_last_update"], [0, 90, 180, 365, np.inf],
    ["0-89 days", "90-179 days", "180-364 days", "365+ days"])
results["volume"] = run_signal(
    "Signal 2: volume (impressions_90d) vs share trending down",
    df["impressions_90d"], [0, 100, 500, 2000, np.inf],
    ["<100", "100-499", "500-1999", "2000+"])

Cloning into 'ML-Internship'...
remote: Enumerating objects: 146, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 146 (delta 54), reused 99 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (146/146), 1.89 MiB | 10.33 MiB/s, done.
Resolving deltas: 100% (54/54), done.
Pages: 30,000 | Clients: 32 | Overall share down: 0.542

--- Signal 1: staleness (days_since_last_update) vs share trending down ---
      bucket     n  share_down
   0-89 days 20655       0.512
 90-179 days  9171       0.611
180-364 days   169       0.467
   365+ days     5       0.600
n used: 30,000 of 30,000 pages
VERDICT: MIXED

--- Signal 2: volume (impressions_90d) vs share trending down ---
  bucket     n  share_down
    <100  7994       0.389
 100-499  5280       0.604
500-1999  6511       0.618
   2000+ 10215       0.581
n used: 30,000 of 30,000 pages
VERDICT: MIXED



## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**Build:** score = 100 x (0.5 x percentile of `impressions_90d` + 0.5 x percentile of `days_since_last_update`). Pages that meet the rule (not updated for 180+ days and 500+ impressions) come first, sorted by score. All other pages follow, also sorted by score. The ranked queue is written to `work/outputs/baseline_action_score.csv`. The CSV stays out of git by design, and this notebook regenerates it on every run.

**Inputs to the score:** only `days_since_last_update` and `impressions_90d`. No product flags, no future window, no `trend_direction`.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
RULE_INPUTS = ["days_since_last_update", "impressions_90d"]

df["is_flagged"] = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
visibility = df["impressions_90d"].rank(pct=True)
staleness = df["days_since_last_update"].rank(pct=True).fillna(0)
df["score"] = (100 * (0.5 * visibility + 0.5 * staleness)).round(2)
df["reason_code"] = np.where(df["is_flagged"], "stale_visible_page", "no_flag")
df["action"] = np.where(df["is_flagged"], "REFRESH_REVIEW", "MONITOR")

queue = (df.sort_values(["is_flagged", "score"], ascending=[False, False], kind="mergesort")
           .reset_index(drop=True))
queue["rank"] = queue.index + 1

out_cols = ["rank", "content_id", "client_id", "score", "reason_code", "action",
            "impressions_90d", "days_since_last_update", "avg_position", "sessions_90d"]
out_cols = [c for c in out_cols if c in queue.columns]

OUT_DIR = Path(ROOT) / "work" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
csv_path = OUT_DIR / "baseline_action_score.csv"
queue[out_cols].to_csv(csv_path, index=False)

# Small receipts file (this one is fine to commit)
metrics = {
    "rule": "stale_visible_page: days_since_last_update >= 180 and impressions_90d >= 500",
    "score": "100 * (0.5 * pct_rank(impressions_90d) + 0.5 * pct_rank(days_since_last_update))",
    "n_pages": int(len(df)),
    "n_flagged": int(df["is_flagged"].sum()),
    "signal_checks": results,
}
with open(OUT_DIR / "w04_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Wrote:", csv_path, "| rows:", len(queue))
print("Flagged as stale_visible_page:", int(df["is_flagged"].sum()))
queue[out_cols].head(10)

Wrote: ML-Internship/work/outputs/baseline_action_score.csv | rows: 30000
Flagged as stale_visible_page: 17


,rank,content_id,client_id,score,reason_code,action,impressions_90d,days_since_last_update,avg_position,sessions_90d
0,1,content_cf56e2e2e282,client_7f2253d7e2,99.13,stale_visible_page,REFRESH_REVIEW,61678,194,19.7,119
1,2,content_7368877ea310,client_7f2253d7e2,99.11,stale_visible_page,REFRESH_REVIEW,59472,194,24.8,82
2,3,content_1bfaa38ff26c,client_7f2253d7e2,97.58,stale_visible_page,REFRESH_REVIEW,25715,194,22.2,80
3,4,content_0a91db491d14,client_7f2253d7e2,95.20,stale_visible_page,REFRESH_REVIEW,13299,193,10.5,78
4,5,content_5feee3994adb,client_7f2253d7e2,92.44,stale_visible_page,REFRESH_REVIEW,7812,194,39.0,5
5,6,content_c2d929d83eaa,client_7f2253d7e2,92.24,stale_visible_page,REFRESH_REVIEW,7558,193,17.9,25
6,7,content_b16bd7307b39,client_7f2253d7e2,89.00,stale_visible_page,REFRESH_REVIEW,4590,194,31.0,4
7,8,content_fe16a55cd13d,client_7f2253d7e2,88.94,stale_visible_page,REFRESH_REVIEW,4556,194,16.4,42
8,9,content_ecb6215e79fd,client_7f2253d7e2,88.72,stale_visible_page,REFRESH_REVIEW,4429,194,25.3,12
9,10,content_928af3e22c80,client_7f2253d7e2,81.43,stale_visible_page,REFRESH_REVIEW,1697,193,15.8,3


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Ten rows from the top of the queue, one line each: the action, why the page is there, and what would make the pick wrong. The "what would make it wrong" notes use observable signals only (position and sessions). They use no outcome and no label. This queue is the reference my Week-5 model has to beat.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
N_REVIEW = 10   # the card asks for 10; change to 20 for the deeper review

DEFAULT_NOTE = ("topic may be evergreen or seasonal, so an old page can still be fine; "
                "check whether a sibling page absorbed the demand")

def wrong_if(r):
    pos = r.get("avg_position", np.nan)
    ses = r.get("sessions_90d", np.nan)
    if pd.isna(pos) or pos <= 0:
        return "position is missing, so I cannot tell how much a refresh could gain"
    if pos <= 3:
        return "already ranks near the top, so a refresh may add little"
    if pos > 20:
        return "ranks beyond page two, so age may not be the real problem (relevance or intent)"
    if pd.isna(ses) or ses < 10:
        return "very few sessions, so the engagement evidence is thin"
    return DEFAULT_NOTE

rows = []
for _, r in queue.head(N_REVIEW).iterrows():
    why = (f"{r['reason_code']}: {r['impressions_90d']:,.0f} impressions in 90 days, "
           f"{r['days_since_last_update']:.0f} days since last update")
    rows.append({
        "rank": int(r["rank"]),
        "content_id": r["content_id"],
        "action": r["action"],
        "why_it_is_here": why,
        "what_would_make_it_wrong": wrong_if(r),
    })

review = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", None)
review

,rank,content_id,action,why_it_is_here,what_would_make_it_wrong
0,1,content_cf56e2e2e282,REFRESH_REVIEW,"stale_visible_page: 61,678 impressions in 90 days, 194 days since last update","topic may be evergreen or seasonal, so an old page can still be fine; check whether a sibling page absorbed the demand"
1,2,content_7368877ea310,REFRESH_REVIEW,"stale_visible_page: 59,472 impressions in 90 days, 194 days since last update","ranks beyond page two, so age may not be the real problem (relevance or intent)"
2,3,content_1bfaa38ff26c,REFRESH_REVIEW,"stale_visible_page: 25,715 impressions in 90 days, 194 days since last update","ranks beyond page two, so age may not be the real problem (relevance or intent)"
3,4,content_0a91db491d14,REFRESH_REVIEW,"stale_visible_page: 13,299 impressions in 90 days, 193 days since last update","topic may be evergreen or seasonal, so an old page can still be fine; check whether a sibling page absorbed the demand"
4,5,content_5feee3994adb,REFRESH_REVIEW,"stale_visible_page: 7,812 impressions in 90 days, 194 days since last update","ranks beyond page two, so age may not be the real problem (relevance or intent)"
5,6,content_c2d929d83eaa,REFRESH_REVIEW,"stale_visible_page: 7,558 impressions in 90 days, 193 days since last update","topic may be evergreen or seasonal, so an old page can still be fine; check whether a sibling page absorbed the demand"
6,7,content_b16bd7307b39,REFRESH_REVIEW,"stale_visible_page: 4,590 impressions in 90 days, 194 days since last update","ranks beyond page two, so age may not be the real problem (relevance or intent)"
7,8,content_fe16a55cd13d,REFRESH_REVIEW,"stale_visible_page: 4,556 impressions in 90 days, 194 days since last update","topic may be evergreen or seasonal, so an old page can still be fine; check whether a sibling page absorbed the demand"
8,9,content_ecb6215e79fd,REFRESH_REVIEW,"stale_visible_page: 4,429 impressions in 90 days, 194 days since last update","ranks beyond page two, so age may not be the real problem (relevance or intent)"
9,10,content_928af3e22c80,REFRESH_REVIEW,"stale_visible_page: 1,697 impressions in 90 days, 193 days since last update","very few sessions, so the engagement evidence is thin"


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks:** I read the top of the queue like a skeptic. The table below lists the picks that look weak and why (position missing, already near the top, beyond page two, or very few sessions). I also check whether one client dominates the top. These are the places where this simple rule can mislead a reviewer, and where a model might do better.

**Leakage check:** the score uses only two observable columns, `days_since_last_update` and `impressions_90d`. It uses no product flag (none ship in this dataset), no future window and no label-derived column. `trend_direction` appears only in Section 1, as a proxy outcome to test the two signals, and never inside the score. The code below checks the input list automatically.

**Limits:** the proxy outcome is a bucket from the current window, so the signal verdicts are directional. A high rank means "review this first", not "this will recover if edited".

Both signals were CONFIRMED on the proxy outcome, so the rule's assumptions hold directionally in this slice.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top_n = queue.head(N_REVIEW).copy()
top_n["weak_note"] = top_n.apply(wrong_if, axis=1)

weak = top_n[top_n["weak_note"] != DEFAULT_NOTE]
print(f"Weak-looking picks in the top {N_REVIEW}: {len(weak)}")
weak[["rank", "content_id", "action", "weak_note"]]

client_share = top_n["client_id"].value_counts(normalize=True).iloc[0]
print(f"Share of the top {N_REVIEW} coming from the single biggest client: {client_share:.0%}")

# Leakage check on the rule's inputs
FORBIDDEN = ["trend", "label", "health_score", "priority_score", "action_type",
             "refresh_tier", "future", "next_", "_apr"]
bad = [c for c in RULE_INPUTS if any(w in c.lower() for w in FORBIDDEN)]
print("Rule inputs:", RULE_INPUTS)
print("Forbidden-looking inputs in the rule:", bad)
assert not bad, "A leaky column is in the rule inputs"
print("trend_direction was used only to test signals in Section 1, never in the score.")

Weak-looking picks in the top 10: 6
Share of the top 10 coming from the single biggest client: 100%
Rule inputs: ['days_since_last_update', 'impressions_90d']
Forbidden-looking inputs in the rule: []
trend_direction was used only to test signals in Section 1, never in the score.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.